# AdS4 Phase-Lock v20: Probe Sweep

**Goal:** identify which constraints are actually needed for held-out reconstruction.

This notebook extends the v19 setup into a **configuration sweep**.

We keep:
- shared multi-slice backbone
- held-out interpolation test
- observables EE / WL / GEO
- metric reconstruction
- optional derivative-sensitive constraint

and ask:

> which probe sets are sufficient, and which are redundant?

This turns the question from:
- "does it work?"

into:
- **"what is minimally required?"**

## Notebook status

This is a **repo-ready v20 scaffold**.

It uses synthetic placeholder targets so the notebook runs immediately.
Replace the target functions with exact repo targets when moving from scaffold to experiment.

### v20 sweep configurations
We test:
- single observables
- observable pairs
- all observables
- the same sets with derivative matching turned on

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Hyperparameters

In [ ]:
# Control slices
c_values = [0.00, 0.16, 0.30]
train_slices = [0.00, 0.30]
heldout_c = 0.16

# Grids
n_x = 160
n_r = 160
x_min, x_max = -2.0, 2.0
r_min, r_max = 0.2, 3.0

# Training
epochs_sweep = 1600
lr = 1e-3

# Architecture
hidden = 64
depth = 3

# Shared weights
w_smooth = 1e-4
w_consistency = 0.5
w_metric = 1.0
w_obs = 1.0
w_deriv = 0.3

print_every = 200

## Synthetic targets

In [ ]:
def true_metric(r, c):
    return 1.0 + r**2 - (1.0 + 0.8*c) / (r + 0.25) + 0.06 * c * np.sin(2.5 * r)

def true_ee(x, c):
    return np.log(1.0 + x**2) + 0.10 * c * np.exp(-x**2) + 0.02 * np.cos(2*x)

def true_wl(x, c):
    return np.sqrt(1.0 + x**2) + 0.08 * c / (1.0 + x**2) + 0.015 * np.sin(3*x)

def true_geo(x, c):
    return 1.0 / (1.0 + x**2) + 0.12 * c * x**2 / (1.0 + x**2) + 0.01 * np.cos(4*x)

## Data assembly

In [ ]:
x_grid = np.linspace(x_min, x_max, n_x).astype(np.float32)
r_grid = np.linspace(r_min, r_max, n_r).astype(np.float32)

targets = {}
for c in c_values:
    targets[c] = {
        "x": torch.tensor(x_grid, device=device).unsqueeze(1),
        "r": torch.tensor(r_grid, device=device).unsqueeze(1),
        "ee": torch.tensor(true_ee(x_grid, c), device=device).unsqueeze(1),
        "wl": torch.tensor(true_wl(x_grid, c), device=device).unsqueeze(1),
        "geo": torch.tensor(true_geo(x_grid, c), device=device).unsqueeze(1),
        "metric": torch.tensor(true_metric(r_grid, c), device=device).unsqueeze(1),
    }

print("train slices:", train_slices, "| held-out:", heldout_c)

## Model

We keep the v18.1 / v19 shared-backbone structure.

In [ ]:
class SharedBackbone(nn.Module):
    def __init__(self, in_dim=2, hidden=64, depth=3):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

class SliceHead(nn.Module):
    def __init__(self, hidden=64, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, h):
        return self.net(h)

class MultiSliceModel(nn.Module):
    def __init__(self, hidden=64, depth=3):
        super().__init__()
        self.backbone_obs = SharedBackbone(in_dim=2, hidden=hidden, depth=depth)
        self.backbone_metric = SharedBackbone(in_dim=2, hidden=hidden, depth=depth)

        self.ee_head = SliceHead(hidden, 1)
        self.wl_head = SliceHead(hidden, 1)
        self.geo_head = SliceHead(hidden, 1)
        self.metric_head = SliceHead(hidden, 1)

    def forward_obs(self, x, c):
        c_tensor = torch.full_like(x, float(c))
        z = torch.cat([x, c_tensor], dim=1)
        h = self.backbone_obs(z)
        return self.ee_head(h), self.wl_head(h), self.geo_head(h)

    def forward_metric(self, r, c):
        c_tensor = torch.full_like(r, float(c))
        z = torch.cat([r, c_tensor], dim=1)
        h = self.backbone_metric(z)
        return self.metric_head(h)

## Loss helpers

In [ ]:
mse = nn.MSELoss()

def smoothness_loss(y):
    return ((y[1:] - y[:-1])**2).mean()

def finite_difference(y, x):
    dy = y[1:] - y[:-1]
    dx = x[1:] - x[:-1]
    return dy / (dx + 1e-12)

def derivative_match_loss(pred_y, true_y, x):
    pred_d = finite_difference(pred_y, x)
    true_d = finite_difference(true_y, x)
    return mse(pred_d, true_d)

def backbone_consistency_loss(model, x_probe, c1, c2):
    c1_tensor = torch.full_like(x_probe, float(c1))
    c2_tensor = torch.full_like(x_probe, float(c2))
    z1 = torch.cat([x_probe, c1_tensor], dim=1)
    z2 = torch.cat([x_probe, c2_tensor], dim=1)
    h1 = model.backbone_obs(z1)
    h2 = model.backbone_obs(z2)
    return ((h1 - h2)**2).mean()

## Sweep configurations

Each config picks:
- which observables are active
- whether derivative matching is included

In [ ]:
configs = [
    {"name": "EE", "obs": ["ee"], "use_deriv": False},
    {"name": "WL", "obs": ["wl"], "use_deriv": False},
    {"name": "GEO", "obs": ["geo"], "use_deriv": False},
    {"name": "EE+WL", "obs": ["ee", "wl"], "use_deriv": False},
    {"name": "EE+GEO", "obs": ["ee", "geo"], "use_deriv": False},
    {"name": "WL+GEO", "obs": ["wl", "geo"], "use_deriv": False},
    {"name": "EE+WL+GEO", "obs": ["ee", "wl", "geo"], "use_deriv": False},
    {"name": "EE+d", "obs": ["ee"], "use_deriv": True},
    {"name": "WL+d", "obs": ["wl"], "use_deriv": True},
    {"name": "GEO+d", "obs": ["geo"], "use_deriv": True},
    {"name": "EE+WL+d", "obs": ["ee", "wl"], "use_deriv": True},
    {"name": "EE+GEO+d", "obs": ["ee", "geo"], "use_deriv": True},
    {"name": "WL+GEO+d", "obs": ["wl", "geo"], "use_deriv": True},
    {"name": "EE+WL+GEO+d", "obs": ["ee", "wl", "geo"], "use_deriv": True},
]
len(configs)

## Config-aware loss

In [ ]:
def compute_config_loss(model, batch, c, obs_names, use_deriv):
    pred_ee, pred_wl, pred_geo = model.forward_obs(batch["x"], c)
    pred_metric = model.forward_metric(batch["r"], c)

    pred_map = {
        "ee": pred_ee,
        "wl": pred_wl,
        "geo": pred_geo,
    }

    total = w_metric * mse(pred_metric, batch["metric"])
    obs_losses = {}
    deriv_losses = {}

    for name in ["ee", "wl", "geo"]:
        if name in obs_names:
            obs_loss = mse(pred_map[name], batch[name])
            total = total + w_obs * obs_loss
            obs_losses[name] = float(obs_loss.item())
            if use_deriv:
                d_loss = derivative_match_loss(pred_map[name], batch[name], batch["x"])
                total = total + w_deriv * d_loss
                deriv_losses[name] = float(d_loss.item())
            else:
                deriv_losses[name] = np.nan
        else:
            obs_losses[name] = np.nan
            deriv_losses[name] = np.nan

    metric_smooth = smoothness_loss(pred_metric)
    total = total + w_smooth * metric_smooth

    if use_deriv:
        d_metric = derivative_match_loss(pred_metric, batch["metric"], batch["r"])
        total = total + w_deriv * d_metric
        deriv_metric_val = float(d_metric.item())
    else:
        deriv_metric_val = np.nan

    parts = {
        "metric": float(mse(pred_metric, batch["metric"]).item()),
        "smooth": float(metric_smooth.item()),
        "d_metric": deriv_metric_val,
        "ee": obs_losses["ee"],
        "wl": obs_losses["wl"],
        "geo": obs_losses["geo"],
        "d_ee": deriv_losses["ee"],
        "d_wl": deriv_losses["wl"],
        "d_geo": deriv_losses["geo"],
        "total": float(total.item()),
    }
    return total, parts

## Train one config

In [ ]:
def train_config(config, c_train, epochs=epochs_sweep, lr=lr, hidden=hidden, depth=depth, verbose=False):
    model = MultiSliceModel(hidden=hidden, depth=depth).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    history = {"total": [], "consistency": []}
    x_probe = torch.linspace(-1.5, 1.5, 100, device=device).unsqueeze(1)

    for epoch in range(epochs):
        opt.zero_grad()

        total_loss = 0.0
        for c in c_train:
            loss_c, _ = compute_config_loss(model, targets[c], c, config["obs"], config["use_deriv"])
            total_loss = total_loss + loss_c

        consistency = 0.0
        for i in range(len(c_train) - 1):
            consistency = consistency + backbone_consistency_loss(model, x_probe, c_train[i], c_train[i+1])

        total_loss = total_loss + w_consistency * consistency
        total_loss.backward()
        opt.step()

        history["total"].append(float(total_loss.item()))
        history["consistency"].append(float(consistency.item()) if torch.is_tensor(consistency) else float(consistency))

        if verbose and (epoch % print_every == 0 or epoch == epochs - 1):
            print(f"[{config['name']}] epoch {epoch:4d} | total {history['total'][-1]:.6e} | consistency {history['consistency'][-1]:.6e}")

    return model, history

## Evaluation helpers

In [ ]:
def metric_mse(model, c):
    with torch.no_grad():
        pred = model.forward_metric(targets[c]["r"], c)
        return float(mse(pred, targets[c]["metric"]).item())

def metric_deriv_mse(model, c):
    with torch.no_grad():
        pred = model.forward_metric(targets[c]["r"], c)
        pred_d = finite_difference(pred, targets[c]["r"])
        true_d = finite_difference(targets[c]["metric"], targets[c]["r"])
        return float(mse(pred_d, true_d).item())

def observable_eval(model, c):
    with torch.no_grad():
        pred_ee, pred_wl, pred_geo = model.forward_obs(targets[c]["x"], c)
        return {
            "ee": float(mse(pred_ee, targets[c]["ee"]).item()),
            "wl": float(mse(pred_wl, targets[c]["wl"]).item()),
            "geo": float(mse(pred_geo, targets[c]["geo"]).item()),
        }

def observable_avg_for_active(eval_dict, obs_names):
    vals = [eval_dict[name] for name in obs_names]
    return float(np.mean(vals)) if len(vals) > 0 else np.nan

## Run the sweep

In [ ]:
results = []
trained_models = {}
histories = {}

for i, config in enumerate(configs, start=1):
    print(f"\n[{i}/{len(configs)}] Training config: {config['name']}")
    model, history = train_config(config, train_slices, verbose=False)
    trained_models[config["name"]] = model
    histories[config["name"]] = history

    held_metric = metric_mse(model, heldout_c)
    held_dmetric = metric_deriv_mse(model, heldout_c)
    held_obs = observable_eval(model, heldout_c)
    held_obs_avg = observable_avg_for_active(held_obs, config["obs"])

    row = {
        "config": config["name"],
        "obs": ",".join(config["obs"]),
        "use_deriv": config["use_deriv"],
        "held_metric_mse": held_metric,
        "held_metric_deriv_mse": held_dmetric,
        "held_obs_avg_mse": held_obs_avg,
        "held_ee_mse": held_obs["ee"],
        "held_wl_mse": held_obs["wl"],
        "held_geo_mse": held_obs["geo"],
        "final_train_loss": history["total"][-1],
        "final_consistency": history["consistency"][-1],
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values(["held_metric_mse", "held_metric_deriv_mse", "held_obs_avg_mse"])
results_df

## Ranked table

In [ ]:
results_df.reset_index(drop=True)

## Best configs by metric MSE

In [ ]:
results_df[["config", "held_metric_mse", "held_metric_deriv_mse", "held_obs_avg_mse"]].head(10)

## Best configs by derivative MSE

In [ ]:
results_df.sort_values("held_metric_deriv_mse")[["config", "held_metric_mse", "held_metric_deriv_mse", "held_obs_avg_mse"]].head(10)

## Bar chart: metric MSE across configs

In [ ]:
plot_df = results_df.sort_values("held_metric_mse")

plt.figure(figsize=(12, 6))
plt.bar(plot_df["config"], plot_df["held_metric_mse"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("held-out metric MSE")
plt.title("v20 probe sweep: held-out metric error by config")
plt.tight_layout()
plt.show()

## Bar chart: derivative MSE across configs

In [ ]:
plot_df = results_df.sort_values("held_metric_deriv_mse")

plt.figure(figsize=(12, 6))
plt.bar(plot_df["config"], plot_df["held_metric_deriv_mse"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("held-out metric derivative MSE")
plt.title("v20 probe sweep: held-out metric derivative error by config")
plt.tight_layout()
plt.show()

## Scatter plot: value fit vs shape fit

In [ ]:
plt.figure(figsize=(8, 6))
for _, row in results_df.iterrows():
    plt.scatter(row["held_metric_mse"], row["held_metric_deriv_mse"])
    plt.text(row["held_metric_mse"], row["held_metric_deriv_mse"], row["config"], fontsize=8)

plt.xlabel("held-out metric MSE")
plt.ylabel("held-out metric derivative MSE")
plt.title("v20: value fit vs shape fit across probe sets")
plt.tight_layout()
plt.show()

## Top configuration diagnostics

In [ ]:
best_config_name = results_df.iloc[0]["config"]
best_model = trained_models[best_config_name]
best_row = results_df.iloc[0]

print("Best config:", best_config_name)
print(best_row)

In [ ]:
x = targets[heldout_c]["x"].detach().cpu().numpy().flatten()
r = targets[heldout_c]["r"].detach().cpu().numpy().flatten()

with torch.no_grad():
    pred_ee, pred_wl, pred_geo = best_model.forward_obs(targets[heldout_c]["x"], heldout_c)
    pred_metric = best_model.forward_metric(targets[heldout_c]["r"], heldout_c)

plt.figure(figsize=(9, 5))
plt.plot(x, targets[heldout_c]["ee"].detach().cpu().numpy(), label="EE true")
plt.plot(x, pred_ee.detach().cpu().numpy(), "--", label="EE pred")
plt.plot(x, targets[heldout_c]["wl"].detach().cpu().numpy(), label="WL true")
plt.plot(x, pred_wl.detach().cpu().numpy(), "--", label="WL pred")
plt.plot(x, targets[heldout_c]["geo"].detach().cpu().numpy(), label="GEO true")
plt.plot(x, pred_geo.detach().cpu().numpy(), "--", label="GEO pred")
plt.xlabel("x")
plt.ylabel("observable")
plt.title(f"Best config held-out observable fit: {best_config_name}")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(r, targets[heldout_c]["metric"].detach().cpu().numpy(), label="metric true")
plt.plot(r, pred_metric.detach().cpu().numpy(), "--", label="metric pred")
plt.xlabel("r")
plt.ylabel("metric")
plt.title(f"Best config held-out metric fit: {best_config_name}")
plt.legend()
plt.show()

In [ ]:
with torch.no_grad():
    true_d = finite_difference(targets[heldout_c]["metric"], targets[heldout_c]["r"]).detach().cpu().numpy().flatten()
    pred_d = finite_difference(pred_metric, targets[heldout_c]["r"]).detach().cpu().numpy().flatten()

r_mid = targets[heldout_c]["r"].detach().cpu().numpy().flatten()[1:]

plt.figure(figsize=(8, 5))
plt.plot(r_mid, true_d, label="metric derivative true")
plt.plot(r_mid, pred_d, "--", label="metric derivative pred")
plt.xlabel("r")
plt.ylabel("d(metric)/dr")
plt.title(f"Best config held-out derivative fit: {best_config_name}")
plt.legend()
plt.show()

## Interpretation guide

### Strong outcome
If a smaller subset beats the full stack, that suggests:
- some probes are redundant
- a minimal identifying set exists

### Alternative strong outcome
If only the full stack with derivative performs best, that suggests:
- the full combined constraint system is genuinely necessary

Either way, v20 sharpens the scientific claim from:
- "a constraint helps"

to:
- **"this is the minimal or necessary constraint structure."**

## Suggested next notebooks

- **v20.1**: ablation of derivative weight  
- **v20.2**: noise robustness sweep  
- **v21**: seed robustness and uncertainty bands  
- **v22**: swap scaffold targets for repo-native physics targets

## Final repo note

When moving from scaffold to real experiment:

1. replace synthetic targets with repo-native observables  
2. export the ranked results table  
3. save the bar chart + scatter plot as README-ready images  
4. compare best v20 config directly against v19 baseline  
5. summarize whether the best set is minimal, necessary, or redundant